# مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ ہیومن ان دی لوپ ورک فلو

## 🎯 تعلیمی مقاصد

اس نوٹ بک میں، آپ سیکھیں گے کہ کیسے Microsoft Agent Framework کے `RequestInfoExecutor` کا استعمال کرتے ہوئے **ہیومن ان دی لوپ** ورک فلو کو نافذ کیا جاتا ہے۔ یہ طاقتور پیٹرن آپ کو مصنوعی ذہانت کے ورک فلو کو روک کر انسانی ان پٹ حاصل کرنے کی اجازت دیتا ہے، جس سے آپ کے ایجنٹس انٹرایکٹو بن جاتے ہیں اور انسان اہم فیصلوں پر قابو رکھتے ہیں۔

## 🔄 ہیومن ان دی لوپ کیا ہے؟

**ہیومن ان دی لوپ (HITL)** ایک ڈیزائن پیٹرن ہے جہاں AI ایجنٹس عمل کو روک کر انسانی ان پٹ طلب کرتے ہیں پھر جاری رکھتے ہیں۔ یہ اس لیے ضروری ہے کہ:

- ✅ **اہم فیصلے** - اہم کارروائیوں سے پہلے انسانی منظوری حاصل کریں
- ✅ **غیر واضح حالات** - جب AI کو یقین نہ ہو تو انسانوں سے وضاحت لیں
- ✅ **صارف کی ترجیحات** - صارفین سے متعدد اختیارات میں سے انتخاب کروائیں
- ✅ **مطابقت اور حفاظت** - منظم آپریشنز کے لیے انسانی نگرانی کو یقینی بنائیں
- ✅ **انٹرایکٹو تجربات** - بات چیت کرنے والے ایجنٹس بنائیں جو صارف کے ان پٹ کا جواب دیں

## 🏗️ یہ مائیکروسافٹ ایجنٹ فریم ورک میں کیسے کام کرتا ہے

فریم ورک HITL کے لیے تین اہم اجزاء فراہم کرتا ہے:

1. **`RequestInfoExecutor`** - ایک خاص ایگزیکیوٹر جو ورک فلو کو روک کر `RequestInfoEvent` جاری کرتا ہے
2. **`RequestInfoMessage`** - انسانوں کو بھیجی جانے والی ٹائپڈ درخواست کے پے لوڈ کا بنیادی کلاس
3. **`RequestResponse`** - `request_id` کا استعمال کرتے ہوئے انسانی جوابات کو اصل درخواستوں سے منسلک کرتا ہے

**ورک فلو پیٹرن:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 ہمارا مثال: صارف کی تصدیق کے ساتھ ہوٹل بکنگ

ہم شرطی ورک فلو کو بڑھائیں گے اور متبادل مقامات تجویز کرنے سے **پہلے** انسانی تصدیق شامل کریں گے:

1. صارف ایک مقام کی درخواست کرتا ہے (مثلاً "پیرس")
2. `availability_agent` چیک کرتا ہے کہ کمروں کی دستیابی ہے یا نہیں
3. **اگر کمرا نہ ہو** → `confirmation_agent` پوچھتا ہے "کیا آپ متبادل دیکھنا چاہتے ہیں؟"
4. ورک فلو `RequestInfoExecutor` کے ذریعے **رک جاتا ہے**
5. **انسان جواب دیتا ہے** "ہاں" یا "نہیں" کنسول ان پٹ کے ذریعے
6. `decision_manager` جواب کی بنیاد پر راستہ منتخب کرتا ہے:
   - **ہاں** → متبادل مقامات دکھائیں
   - **نہیں** → بکنگ کی درخواست منسوخ کریں
7. حتمی نتیجہ دکھائیں

یہ دکھاتا ہے کہ صارفین کو ایجنٹ کی تجاویز پر قابو کیسے دیا جائے!

---

آئیے شروع کریں! 🚀


## مرحلہ 1: مطلوبہ لائبریریز درآمد کریں

ہم معیاری ایجنٹ فریم ورک کے اجزاء کے ساتھ **human-in-the-loop مخصوص کلاسز** درآمد کرتے ہیں:
- `RequestInfoExecutor` - ایگزیکیوٹر جو انسانی ان پٹ کے لئے ورک فلو کو روکتا ہے
- `RequestInfoEvent` - واقعہ جو انسانی ان پٹ کی درخواست پر بھیجا جاتا ہے
- `RequestInfoMessage` - ٹائپ شدہ درخواست کے پے لوڈز کے لیے بنیادی کلاس
- `RequestResponse` - انسانی جوابات کو درخواستوں کے ساتھ مربوط کرتا ہے
- `WorkflowOutputEvent` - ورک فلو آؤٹ پٹ کا پتہ لگانے کے لیے واقعہ


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## قدم 2: منظم آؤٹ پٹس کے لیے Pydantic ماڈلز کی تعریف کریں

یہ ماڈلز وہ **schema** تعریف کرتے ہیں جو ایجنٹس واپس کریں گے۔ ہم مشروط ورک فلو کے تمام ماڈلز کو رکھتے ہیں اور شامل کرتے ہیں:

**Human-in-the-Loop کے لیے نیا:**
- `HumanFeedbackRequest` - `RequestInfoMessage` کا سب کلاس جو انسانوں کو بھیجی جانے والی درخواست کا مواد تعریف کرتا ہے
  - اس میں `prompt` (پوچھنے کے لیے سوال) اور `destination` (غیر دستیاب شہر کے بارے میں سیاق و سباق) شامل ہیں


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## مرحلہ 3: ہوٹل بکنگ کا آلہ بنائیں

وہی آلہ جو مشروط ورک فلو سے ہے - یہ چیک کرتا ہے کہ منزل پر کمرے دستیاب ہیں یا نہیں۔


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## مرحلہ 4: راستہ بندی کے لیے حالت کی افعال کی تعریف کریں

ہمیں اپنے انسانی دخل کے ورک فلو کے لیے **چار حالت کی افعال** کی ضرورت ہے:

**مشروط ورک فلو سے:**
1. `has_availability_condition` - جب ہوٹل دستیاب ہوں تو راستہ دیتا ہے
2. `no_availability_condition` - جب ہوٹل دستیاب نہ ہوں تو راستہ دیتا ہے

**انسانی دخل کے لیے نیا:**
3. `user_wants_alternatives_condition` - جب صارف متبادل کو "ہاں" کہے تو راستہ دیتا ہے
4. `user_declines_alternatives_condition` - جب صارف متبادل کو "نہیں" کہے تو راستہ دیتا ہے


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## مرحلہ 5: فیصلہ ساز مینیجر ایگزیکییوٹر بنائیں

یہ **انسانی-اندر-لوپ پیٹرن کا مرکزی حصہ** ہے! `DecisionManager` ایک مخصوص `Executor` ہے جو:

1. **انسانی رائے وصول کرتا ہے** `RequestResponse` آبجیکٹس کے ذریعے
2. **صارف کے فیصلے کو پروسیس کرتا ہے** (ہاں/نہیں)
3. **ورک فلو کو راہنمائی دیتا ہے** مناسب ایجنٹس کو پیغامات بھیج کر

اہم خصوصیات:
- `@handler` ڈیکوریٹر استعمال کرتا ہے تاکہ طریقے ورک فلو کے مراحل کے طور پر ظاہر ہوں
- `RequestResponse[HumanFeedbackRequest, str]` وصول کرتا ہے جس میں اصل درخواست اور صارف کا جواب دونوں ہوتے ہیں
- سادہ "ہاں" یا "نہیں" پیغامات دیتا ہے جو ہمارے شرطی فنکشنز کو چالو کرتے ہیں


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## مرحلہ 6: کسٹم ڈسپلے ایگزیکیوٹر بنائیں

وہی ڈسپلے ایگزیکیوٹر شرطی ورک فلو سے - ورک فلو کے آؤٹ پٹ کے طور پر حتمی نتائج دیتا ہے۔


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## مرحلہ 7: ماحولياتي متغيرات لوڈ کریں

LLM کلائنٹ (Microsoft Foundry / Azure OpenAI) کو ترتیب دیں۔


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## مرحلہ 8: AI ایجنٹس اور ایگزیکیوٹرز بنائیں

ہم **چھ ورک فلو اجزاء** بناتے ہیں:

**ایجنٹس (AgentExecutor میں لپٹے ہوئے):**
1. **availability_agent** - ٹول استعمال کرتے ہوئے ہوٹل کی دستیابی چیک کرتا ہے
2. **confirmation_agent** - 🆕 انسانی تصدیقی درخواست تیار کرتا ہے
3. **alternative_agent** - متبادل شہروں کی تجویز دیتا ہے (جب صارف ہاں کہے)
4. **booking_agent** - بکنگ کی ترغیب دیتا ہے (جب کمرے دستیاب ہوں)
5. **cancellation_agent** - 🆕 منسوخی پیغام سنبھالتا ہے (جب صارف نہیں کہے)

**خاص ایگزیکیوٹرز:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` جو انسانی ان پٹ کے لئے ورک فلو کو روک دیتا ہے
7. **decision_manager** - 🆕 کسٹم ایگزیکیوٹر جو انسانی جواب کی بنیاد پر راستہ متعین کرتا ہے (جو پہلے ہی اوپر متعین کیا گیا ہے)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## مرحلہ 9: انسانی شامل ہونے والے ورک فلو کی تعمیر

اب ہم **مشروط راہنمائی** کے ساتھ ورک فلو گراف تعمیر کرتے ہیں جس میں انسانی شامل ہونے والا راستہ بھی شامل ہے:

**ورک فلو کی ساخت:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**اہم راستے:**
- `availability_agent → confirmation_agent` (جب کمرے نہیں ہوں)
- `confirmation_agent → prepare_human_request` (قسم کی تبدیلی)
- `prepare_human_request → request_info_executor` (انسان کے لئے توقف)
- `request_info_executor → decision_manager` (ہمیشہ - RequestResponse فراہم کرتا ہے)
- `decision_manager → alternative_agent` (جب صارف "ہاں" کہے)
- `decision_manager → cancellation_agent` (جب صارف "نہیں" کہے)
- `availability_agent → booking_agent` (جب کمرے دستیاب ہوں)
- تمام راستے `display_result` پر ختم ہوتے ہیں


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## مرحلہ 10: ٹیسٹ کیس 1 چلائیں - شہر بغیر دستیابی کے (Paris کے ساتھ انسانی تصدیق)

یہ ٹیسٹ **مکمل انسان-دورانیہ کے چکر** کا مظاہرہ کرتا ہے:

1. پیرس میں ہوٹل کی درخواست کریں
2. availability_agent چیک کرتا ہے → کمرا نہیں ہے
3. confirmation_agent انسانی سوال تخلیق کرتا ہے
4. request_info_executor **ورک فلو کو روک دیتا ہے** اور `RequestInfoEvent` جاری کرتا ہے
5. **درخواست ایونٹ کا پتہ لگاتی ہے اور کنسول میں صارف سے پوچھتی ہے**
6. صارف "ہاں" یا "نہیں" ٹائپ کرتا ہے
7. درخواست `send_responses_streaming()` کے ذریعے جواب بھیجتی ہے
8. decision_manager جواب کی بنیاد پر راستہ معین کرتا ہے
9. آخری نتیجہ دکھایا جاتا ہے

**اہم پیٹرن:**
- پہلی بار کے لیے `workflow.run_stream()` استعمال کریں
- بعد کی بار کے لیے `workflow.send_responses_streaming(pending_responses)` استعمال کریں
- `RequestInfoEvent` سنیں تاکہ معلوم ہو کہ انسان کا جواب کب درکار ہے
- `WorkflowOutputEvent` سنیں تاکہ آخری نتائج حاصل کیے جا سکیں


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## مرحلہ 11: ٹیسٹ کیس 2 چلائیں - شہر دستیابی کے ساتھ (سٹاک ہوم - انسانی ان پٹ کی ضرورت نہیں)

یہ ٹیسٹ اُس **براہ راست راستے** کو ظاہر کرتا ہے جب کمرے دستیاب ہوں:

1. سٹاک ہوم میں ہوٹل کی درخواست
2. availability_agent چیک کرتا ہے → کمرے دستیاب ہیں ✅
3. booking_agent بکنگ کا مشورہ دیتا ہے
4. display_result تصدیق دکھاتا ہے
5. **انسانی ان پٹ کی کوئی ضرورت نہیں!**

جب کمرے دستیاب ہوں تو ورک فلو پوری طرح انسانی عمل کو چھوڑ دیتا ہے۔


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## اہم نکات اور ہومین-ان-دی-لوپ بہترین طریقہ کار

### ✅ آپ نے کیا سیکھا:

#### 1. **RequestInfoExecutor پیٹرن**
مائیکروسافٹ ایجنٹ فریم ورک میں ہومین-ان-دی-لوپ پیٹرن تین کلیدی عناصر استعمال کرتا ہے:
- `RequestInfoExecutor` - ورک فلو کو روک دیتا ہے اور ایونٹس جاری کرتا ہے
- `RequestInfoMessage` - ٹائپڈ درخواست کے پیلوڈز کے لیے بنیادی کلاس (اس کا ذیلی کلاس بنائیں!)
- `RequestResponse` - انسانی جوابات کو اصل درخواستوں سے مربوط کرتا ہے

**اہم فہم:**
- `RequestInfoExecutor` خود ان پٹ جمع نہیں کرتا - یہ صرف ورک فلو کو روکتا ہے
- آپ کے ایپلیکیشن کوڈ کو `RequestInfoEvent` سن کر ان پٹ جمع کرنا ہوگا
- آپ کو `send_responses_streaming()` کو ایک ڈکٹ دینا ہوگا جو `request_id` کو صارف کے جواب سے مائننگ کرتا ہو

#### 2. **اسٹریمنگ ایگزیکیوشن پیٹرن**
```python
# پہلی بار دہرائی
stream = workflow.run_stream(initial_request)

# بعد کی تکراریں (انسانی ان پٹ کے بعد)
stream = workflow.send_responses_streaming(pending_responses)

# ہمیشہ واقعات کو پراسیس کریں
events = [event async for event in stream]
```

#### 3. **ایونٹ پر مبنی آرکیٹیکچر**
مخصوص ایونٹس سن کر ورک فلو کو کنٹرول کریں:
- `RequestInfoEvent` - انسانی ان پٹ کی ضرورت ہے (ورک فلو رکا ہوا ہے)
- `WorkflowOutputEvent` - حتمی نتیجہ دستیاب ہے (ورک فلو مکمل)
- `WorkflowStatusEvent` - حالت کی تبدیلیاں (IN_PROGRESS، IDLE_WITH_PENDING_REQUESTS، وغیرہ)

#### 4. **کسٹم ایگزیکیوٹرز `@handler` کے ساتھ**
`DecisionManager` دکھاتا ہے کہ کس طرح ایگزیکیوٹرز بنائیں جو:
- `@handler` ڈیکوریٹر استعمال کرتے ہوئے طریقے ورک فلو کے مراحل کے طور پر ظاہر کریں
- ٹائپڈ میسجز وصول کریں (مثلاً `RequestResponse[HumanFeedbackRequest, str]`)
- میسجز بھیج کر ورک فلو کو دوسرے ایگزیکیوٹرز تک پہنچائیں
- `WorkflowContext` کے ذریعے سیاق و سباق تک رسائی حاصل کریں

#### 5. **انسانی فیصلوں کے ساتھ مشروط راستہ**
آپ ایسی شرطی فنکشنز تشکیل دے سکتے ہیں جو انسانی جوابات کا جائزہ لیں:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 حقیقی دنیا کی درخواستیں:

1. **منظوری کے ورک فلو**
   - اخراجات کی رپورٹس کی پراسیسنگ سے پہلے مینیجر کی منظوری حاصل کریں
   - خودکار ای میلز بھیجنے سے قبل انسانی جائزہ کی ضرورت
   - اعلیٰ قدر کے لین دین کی تصدیق قبل از عمل

2. **مواد کی نگرانی**
   - مشتبہ مواد کو انسانی جائزے کے لیے نشان زد کریں
   - ماڈیٹرز سے ایج کیسز پر حتمی فیصلہ کروائیں
   - جب AI کا اعتماد کم ہو تو انسانوں کو اس کا علم دیں

3. **کسٹمر سروس**
   - AI کو معمول کے سوالات خودکار طریقے سے سنبھالنے دیں
   - پیچیدہ مسائل کو انسانی نمائندوں تک پہنچائیں
   - صارف سے پوچھیں کہ کیا وہ انسان سے بات کرنا چاہتے ہیں

4. **ڈیٹا پراسیسنگ**
   - مبہم ڈیٹا اندراجات کو حل کرنے کے لیے انسانوں سے پوچھیں
   - غیر واضح دستاویزات کی AI تشریحات کی تصدیق کریں
   - صارفین کو متعدد درست تشریحات میں سے انتخاب کی اجازت دیں

5. **حفاظتی اہمیت کے نظام**
   - ناقابل واپسی کارروائیوں سے قبل انسانی تصدیق کی ضرورت
   - حساس ڈیٹا تک رسائی سے پہلے منظوری حاصل کریں
   - قواعد و ضوابط والی صنعتوں (صحت کی دیکھ بھال، مالیات) میں فیصلوں کی تصدیق کریں

6. **تفاعلی ایجنٹس**
   - گفتگو کرنے والے بوٹس بنائیں جو اضافی سوالات کریں
   - وزرڈز تخلیق کریں جو صارفین کو پیچیدہ عمل سے رہنمائی کریں
   - ایسے ایجنٹس ڈیزائن کریں جو انسانی ساتھ قدم بہ قدم تعاون کریں

### 🔄 موازنہ: ہومین-ان-دی-لوپ کے ساتھ بمقابلہ بغیر

| خصوصیت | مشروط ورک فلو | ہومین-ان-دی-لوپ ورک فلو |
|---------|---------------------|---------------------------|
| **ایگزیکیوشن** | واحد `workflow.run()` | `run_stream()` + `send_responses_streaming()` کے ساتھ لوپ |
| **صارف کی ان پٹ** | کوئی نہیں (مکمل خودکار) | `input()` یا UI کے ذریعے تفاعلی پرامپٹس |
| **اجزاء** | ایجنٹس + ایگزیکیوٹرز | + RequestInfoExecutor + DecisionManager |
| **ایونٹس** | صرف AgentExecutorResponse | RequestInfoEvent، WorkflowOutputEvent، وغیرہ |
| **رکنا** | نہیں رکتا | ورک فلو RequestInfoExecutor پر رکتا ہے |
| **انسانی کنٹرول** | کوئی انسانی کنٹرول نہیں | انسان کلیدی فیصلے کرتے ہیں |
| **استعمال کی صورت** | خودکاری | تعاون اور نگرانی |

### 🚀 جدید پیٹرنز:

#### متعدد انسانی فیصلہ سازی کے پوائنٹس
آپ ایک ہی ورک فلو میں متعدد `RequestInfoExecutor` نوڈز رکھ سکتے ہیں:
```python
.add_edge(agent1, request_info_1)  # پہلا انسانی فیصلہ
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # دوسرا انسانی فیصلہ
.add_edge(decision_manager_2, final_agent)
```

#### ٹائم آؤٹ ہینڈلنگ
انسانی جوابات کے لیے ٹائم آؤٹ نافذ کریں:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # محفوظ اختیار کو بطور ڈیفالٹ منتخب کریں
```

#### عمدہ UI انٹیگریشن
`input()` کی جگہ ویب UI، سلیک، ٹیمز وغیرہ کے ساتھ انٹیگریٹ کریں:
```python
if isinstance(event, RequestInfoEvent):
    # صارف کے منتخب کردہ چینل پر اطلاع بھیجیں
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### مشروط ہومین-ان-دی-لوپ
صرف مخصوص حالات میں انسانی ان پٹ طلب کریں:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # صرف انسانی طرف راہ دکھائیں اگر اعتماد کم ہو یا قیمت زیادہ ہو
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ بہترین طریقہ کار:

1. **ہمیشہ RequestInfoMessage کو ذیلی کلاس بنائیں**
   - ٹائپ سیفٹی اور ویلیڈیشن فراہم کرتا ہے
   - UI رینڈرنگ کے لیے بھرپور سیاق و سباق فعال کرتا ہے
   - ہر درخواست کی نوعیت کو واضح کرتا ہے

2. **وضاحتی پرامپٹس استعمال کریں**
   - جو آپ پوچھ رہے ہیں اس کے متعلق سیاق شامل کریں
   - ہر انتخاب کے نتائج بیان کریں
   - سوالات کو آسان اور واضح رکھیں

3. **غیر متوقع ان پٹ کو سنبھالیں**
   - صارف کے جوابات کی تصدیق کریں
   - غلط ان پٹ کے لیے ڈیفالٹ مہیا کریں
   - واضح غلطی کے پیغامات دیں

4. **Request IDs کو ٹریک کریں**
   - `request_id` اور جوابات کے درمیان رابطہ کا استعمال کریں
   - دستی طور پر حالت کا انتظام کرنے کی کوشش نہ کریں

5. **نان-بلاک کرنے کے لیے ڈیزائن کریں**
   - ان پٹ کا انتظار کرتے ہوئے تھریڈز کو بلاک نہ کریں
   - پورے کوڈ میں async پیٹرنز استعمال کریں
   - ایک ساتھ متعدد ورک فلو انسٹینس سپورٹ کریں

### 📚 متعلقہ تصورات:

- **ایجنٹ مڈل ویئر** - ایجنٹ کالز کو انٹرسپٹ کریں (پچھلا نوٹ بک)
- **ورک فلو اسٹیٹ مینجمنٹ** - رنز کے درمیان ورک فلو کی حالت محفوظ کریں
- **کثیر ایجنٹ تعاون** - ہومین-ان-دی-لوپ کو ایجنٹس کی ٹیموں کے ساتھ ملائیں
- **ایونٹ پر مبنی آرکیٹیکچرز** - ایونٹس کے ساتھ ریئیکٹو سسٹمز بنائیں

---

### 🎓 مبارک ہو!

آپ نے مائیکروسافٹ ایجنٹ فریم ورک کے ساتھ ہومین-ان-دی-لوپ ورک فلو مکمل طور پر سیکھ لیے ہیں! اب آپ جانتے ہیں کہ کیسے:
- ✅ ورک فلو روک کر انسانی ان پٹ جمع کریں
- ✅ RequestInfoExecutor اور RequestInfoMessage استعمال کریں
- ✅ ایونٹس کے ساتھ اسٹریمنگ ایگزیکیوشن سنبھالیں
- ✅ @handler کے ساتھ کسٹم ایگزیکیوٹرز بنائیں
- ✅ انسانی فیصلوں کی بنیاد پر ورک فلو راہنمائی کریں
- ✅ ایسے تفاعلی AI ایجنٹس بنائیں جو انسانوں کے ساتھ تعاون کرتے ہیں

**یہ قابل اعتماد، قابل کنٹرول AI نظاموں کے لیے ایک اہم پیٹرن ہے!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
